# UC2 — Bonds Are Safe Haven Only in Disinflationary Regimes
### The bond/equity negative correlation is regime-dependent, not structural.
**UC2 — Usual Claims Series**

---

## Abstract

The conventional claim — "TLT goes up when stocks crash" — is true, but only inside a specific macroeconomic regime. This project tests whether the equity/bond negative correlation is structural or regime-conditional using two complementary data layers: a century-long annual series (Damodaran 1928–2024) and a modern tradable implementation (SPY/TLT, 2002–present). Results show the negative correlation is concentrated in disinflationary/rate-down regimes and collapses — or reverses — in inflationary ones. 2022 is not an anomaly. It is the regime behaving exactly as predicted.


In [ ]:
# ── Imports & configuration ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 25)

# ── Utility functions ─────────────────────────────────────────────────────────
def cum_return(s):
    """Geometric cumulative return of a return series."""
    return (1 + s).prod() - 1

def max_drawdown(s):
    """Maximum peak-to-trough drawdown of a return series."""
    wealth = (1 + s).cumprod()
    peak   = wealth.cummax()
    dd     = (wealth - peak) / peak
    return dd.min()

print('✓ Imports OK')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')


In [ ]:
# ── Parameters & constants — nothing hardcoded after this cell ───────────────

# ── Date ranges ───────────────────────────────────────────────────────────────
GW_START       = '1926-01-01'   # Goyal-Welch start (monthly)
GW_END         = '2023-12-31'   # last full year in PredictorData2023
ETF_START      = '2002-07-30'   # TLT inception date
ETF_END        = '2026-05-31'   # data pull end (today-ish)
CPI_START      = '1947-01-01'   # FRED CPIAUCSL starts here

# ── Rolling windows ───────────────────────────────────────────────────────────
MA200_WINDOW   = 200     # days — SPY moving average for episode detection
MA200_SHIFT    = 1       # days — no look-ahead bias
ROLL_CORR_L1   = 10      # years — rolling correlation window (Level 1, annual Damodaran)
ROLL_CORR_GW   = 60      # months — (kept for reference; L1 now uses annual data)
ROLL_CORR_ETF  = 252     # days  — rolling correlation window (Level 2)
CPI_ROLL_MEAN  = 36      # months — rolling mean for regime classification
CPI_LAG        = 1       # months — CPI publication lag

# ── Stress episode detection (inherited from Diversification Series) ───────────
ROLL_PEAK      = 252     # days — rolling high lookback
DD_ONSET       = 0.15    # drawdown threshold to enter episode (> 15% from high)
DD_TROUGH      = 0.20    # threshold used in exit logic
RECOVERY       = 0.075   # recovery from trough needed to exit episode

# ── Hypothesis thresholds (pre-declared, never adjusted post-hoc) ─────────────
H1_CORR_THRESH = -0.10   # H1: rolling corr at onset >= -0.10 in inflationary regime
H2_MIN_EPISODES = 2      # minimum inflationary episodes for H2 to be reportable

# ── Portfolio ─────────────────────────────────────────────────────────────────
W_EQUITY       = 0.60    # 60/40 equity weight
W_BOND         = 0.40    # 60/40 bond weight
REBAL_FREQ     = 'M'     # monthly rebalancing

# ── Visual palette ────────────────────────────────────────────────────────────
C_BLUE  = '#4C72B0'   # disinflationary regime / equity
C_RED   = '#C44E52'   # inflationary regime / bonds under stress
C_GREEN = '#55A868'   # safe / recovery
C_ORANGE= '#DD8452'   # neutral / 60/40

REGIME_COLORS = {'Disinflationary': C_BLUE, 'Inflationary': C_RED}
EP_COLORS     = {True: C_RED, False: C_BLUE}   # True = inflationary onset

print('✓ Parameters set')
print(f'  GW range      : {GW_START} → {GW_END}')
print(f'  ETF range     : {ETF_START} → {ETF_END}')
print(f'  Roll corr L1  : {ROLL_CORR_L1}y  |  ETF: {ROLL_CORR_ETF}d')
print(f'  CPI roll mean : {CPI_ROLL_MEAN}m  |  lag: {CPI_LAG}m')
print(f'  DD onset      : {DD_ONSET:.0%}  |  recovery: {RECOVERY:.1%}')
print(f'  H1 threshold  : rolling corr >= {H1_CORR_THRESH} at onset')
print(f'  Portfolio     : {W_EQUITY:.0%} equity / {W_BOND:.0%} bonds, {REBAL_FREQ} rebal')


## Pre-Declared Hypotheses

> These are fixed before any data is loaded or analyzed. Results are reported as SUPPORTED or NOT SUPPORTED. No spin.

| # | Hypothesis | Threshold | Test |
|---|-----------|-----------|------|
| H1 | During equity stress episodes with inflationary onset, the rolling 252-day SPY/TLT correlation at onset is ≥ −0.10 (positive or near-zero) | corr ≥ −0.10 | Median rolling corr at onset, inflationary episodes only |
| H2 | Median TLT return onset→trough is materially lower in inflationary stress episodes than in disinflationary ones | Directional: inflationary median < disinflationary median | Median comparison across episode regimes |
| H3 | (Descriptive) 60/40 max drawdown during inflationary stress episodes is larger than during disinflationary ones | Directional | Mean/median drawdown by regime |

**Classification rule:** Regime at onset = Inflationary if CPI YoY at onset date > 36-month rolling mean of CPI YoY, using data available at that date (no look-ahead). Disinflationary otherwise.


In [ ]:
# ── Directories ──────────────────────────────────────────────────────────────
IMAGES_DIR = Path('../images')
DATA_DIR   = Path('../data')

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Directories ready')
print(f'  Data   : {DATA_DIR.resolve()}')
print(f'  Images : {IMAGES_DIR.resolve()}')


In [ ]:
# ── Load Damodaran historical returns (Level 1) ──────────────────────────────
# Annual S&P 500 and 10-year T.Bond total returns, 1928-2024
# Source: Aswath Damodaran, NYU Stern (damodaran_historical.csv)

DAM_PATH = DATA_DIR / 'damodaran_historical.csv'
if not DAM_PATH.exists():
    raise FileNotFoundError(f'Damodaran data not found at {DAM_PATH}')

dam = pd.read_csv(DAM_PATH)
dam['Year'] = dam['Year'].astype(int)
dam = dam.set_index('Year')
dam.index = pd.to_datetime([f'{y}-12-31' for y in dam.index])
dam.index.name = 'date'
dam.columns = ['eq_ret', 'ltr']
dam = dam.sort_index()

# ── Build annual CPI YoY for regime classification ────────────────────────────
# FRED monthly CPI (cached from Level 2 pipeline) resampled to annual average
# Pre-1947: BLS historical statistics (hardcoded)
CPI_CACHE = DATA_DIR / 'cpi.csv'
if CPI_CACHE.exists():
    cpi_m = pd.read_csv(CPI_CACHE, index_col=0, parse_dates=True).squeeze()
    cpi_ann_fred = cpi_m.resample('YE').mean()
    cpi_yoy_fred = cpi_ann_fred.pct_change()
    cpi_yoy_fred.index = pd.to_datetime([f'{d.year}-12-31' for d in cpi_yoy_fred.index])
else:
    cpi_yoy_fred = pd.Series(dtype=float)

PRE_1947 = {
    1928:-0.012, 1929: 0.000, 1930:-0.023, 1931:-0.090,
    1932:-0.099, 1933:-0.051, 1934: 0.031, 1935: 0.022,
    1936: 0.015, 1937: 0.036, 1938:-0.021, 1939:-0.014,
    1940: 0.007, 1941: 0.107, 1942: 0.090, 1943: 0.030,
    1944: 0.023, 1945: 0.011, 1946: 0.083, 1947: 0.143,
}
pre47_series = pd.Series({pd.Timestamp(f'{y}-12-31'): v for y, v in PRE_1947.items()})
cpi_yoy_annual = pd.concat([
    pre47_series,
    cpi_yoy_fred[cpi_yoy_fred.index > pd.Timestamp('1947-12-31')]
]).sort_index()

dam_cpi_yoy = cpi_yoy_annual.reindex(dam.index, method='nearest')

# Regime: CPI YoY > 3-year rolling mean, shifted 1 year (no look-ahead)
dam_roll3 = dam_cpi_yoy.rolling(3, min_periods=2).mean().shift(1)
dam_regime = (dam_cpi_yoy > dam_roll3).astype(int)   # 1=Inflationary

dam['cpi_yoy'] = dam_cpi_yoy.values
dam['regime']  = dam_regime.values
gw = dam.copy()   # backward-compat alias used in downstream cells
GW_AVAILABLE = True

assert dam.index.is_monotonic_increasing
assert not dam.index.duplicated().any()
assert len(dam) >= 90, f'Expected >=90 years, got {len(dam)}'

print('✓ Damodaran historical data loaded (Level 1)')
print(f'  Range      : {dam.index[0].year} - {dam.index[-1].year}  ({len(dam)} years)')
print(f'  Infl years : {(dam_regime==1).sum()} / {dam_regime.notna().sum()}  ({(dam_regime==1).mean():.1%})')
print()
print(dam[['eq_ret','ltr','cpi_yoy','regime']].tail(10).round(4))


In [ ]:
# ── Download SPY and TLT prices ───────────────────────────────────────────────
# auto_adjust=True: prices adjusted for splits + dividends — returns are total returns
# Cache to data/spy_prices.csv and data/tlt_prices.csv

import yfinance as yf

def load_etf(ticker, cache_path, start, end):
    """Load ETF closing prices with caching."""
    if Path(cache_path).exists():
        s = pd.read_csv(cache_path, index_col=0, parse_dates=True).squeeze()
        print(f'  {ticker}: loaded from cache ({len(s)} rows)')
    else:
        print(f'  {ticker}: downloading ...')
        raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
        s = raw['Close'].squeeze()
        s.to_csv(cache_path)
        print(f'  {ticker}: downloaded and cached ({len(s)} rows)')
    s.name = ticker
    return s.sort_index()

spy_px = load_etf('SPY', DATA_DIR / 'spy_prices.csv', ETF_START, ETF_END)
tlt_px = load_etf('TLT', DATA_DIR / 'tlt_prices.csv', ETF_START, ETF_END)

# Align to common index
common_idx = spy_px.index.intersection(tlt_px.index)
spy_px = spy_px.reindex(common_idx)
tlt_px = tlt_px.reindex(common_idx)

# Daily returns
spy_ret = spy_px.pct_change()
tlt_ret = tlt_px.pct_change()

# ── Asserts ───────────────────────────────────────────────────────────────────
assert spy_px.index.is_monotonic_increasing
assert tlt_px.index.is_monotonic_increasing
assert not spy_px.index.duplicated().any()
assert spy_ret.notna().sum() >= 1000, 'Too few SPY observations'
assert tlt_ret.notna().sum() >= 1000, 'Too few TLT observations'

print('✓ ETF prices loaded')
print(f'  SPY: {spy_px.index[0].date()} → {spy_px.index[-1].date()}  ({len(spy_px)} obs)')
print(f'  TLT: {tlt_px.index[0].date()} → {tlt_px.index[-1].date()}  ({len(tlt_px)} obs)')
print(f'  Common observations: {len(common_idx)}')


In [ ]:
# ── Load CPI (FRED CPIAUCSL) ─────────────────────────────────────────────────
# CPI monthly → YoY → 36m rolling mean → regime flag
# Forward-fill with 1-month publication lag (no look-ahead)

CPI_CACHE = DATA_DIR / 'cpi.csv'

if CPI_CACHE.exists():
    cpi_raw = pd.read_csv(CPI_CACHE, index_col=0, parse_dates=True).squeeze()
    print(f'  CPI: loaded from cache ({len(cpi_raw)} rows)')
else:
    print('  CPI: downloading from FRED ...')
    try:
        import pandas_datareader.data as web
        cpi_raw = web.DataReader('CPIAUCSL', 'fred', CPI_START, ETF_END)['CPIAUCSL']
        cpi_raw.to_csv(CPI_CACHE)
        print(f'  CPI: downloaded and cached ({len(cpi_raw)} rows)')
    except Exception as e:
        print(f'  pandas_datareader failed: {e}')
        print('  Trying direct FRED API ...')
        try:
            import urllib.request, io
            url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL'
            with urllib.request.urlopen(url, timeout=30) as resp:
                cpi_raw = pd.read_csv(io.BytesIO(resp.read()), index_col=0, parse_dates=True).squeeze()
            cpi_raw.name = 'CPIAUCSL'
            cpi_raw.to_csv(CPI_CACHE)
            print(f'  CPI: downloaded via FRED CSV API ({len(cpi_raw)} rows)')
        except Exception as e2:
            raise SystemExit(f'CPI download failed: {e2}')

cpi_raw = cpi_raw.sort_index()

# ── Compute CPI YoY and regime flag ──────────────────────────────────────────
# cpi_yoy: published level → 12m change. Shift 1 month for publication lag.
cpi_monthly = cpi_raw.resample('ME').last()
cpi_yoy = cpi_monthly.pct_change(12).shift(CPI_LAG)   # 1-month pub lag
cpi_roll36 = cpi_yoy.rolling(CPI_ROLL_MEAN, min_periods=18).mean().shift(1)
# shift(1) on rolling mean ensures no look-ahead: we use only past data

cpi_regime_monthly = (cpi_yoy > cpi_roll36).astype(int)  # 1 = Inflationary
cpi_regime_monthly.name = 'regime'  # 1=Inflationary, 0=Disinflationary

# ── Forward-fill to daily ─────────────────────────────────────────────────────
cpi_yoy_daily    = cpi_yoy.reindex(spy_px.index, method='ffill')
cpi_regime_daily = cpi_regime_monthly.reindex(spy_px.index, method='ffill')

# ── Asserts ───────────────────────────────────────────────────────────────────
assert cpi_yoy.notna().sum() >= 200
assert cpi_regime_monthly.notna().sum() >= 200
inflationary_frac = cpi_regime_daily.mean()
assert 0.05 < inflationary_frac < 0.95, f'Regime classification degenerate: {inflationary_frac:.2%}'

print('✓ CPI loaded and regime computed')
print(f'  CPI range   : {cpi_monthly.index[0].date()} → {cpi_monthly.index[-1].date()}')
print(f'  YoY range   : {cpi_yoy.first_valid_index().date()} → {cpi_yoy.last_valid_index().date()}')
print(f'  Inflationary (daily, ETF period): {inflationary_frac:.1%} of trading days')


---
## Block 1 — Long History: Annual Correlation by Regime (Damodaran 1928–2024)

Tests the structural claim: is the equity/bond negative correlation persistent across all regimes or concentrated in specific macroeconomic environments?


In [ ]:
# ── Level 1: Annual rolling 10-year equity/bond correlation by regime ──────────
if not GW_AVAILABLE or gw is None:
    print('SKIPPED: Level 1 data not available.')
    corr_by_regime = None
else:
    roll_corr_l1 = gw['eq_ret'].rolling(ROLL_CORR_L1, min_periods=5).corr(gw['ltr'])
    gw['roll_corr'] = roll_corr_l1

    corr_by_regime = gw.groupby('regime')['roll_corr'].agg(['mean','median','std','count'])
    corr_by_regime.index = ['Disinflationary', 'Inflationary']

    print('Level 1: rolling 10-year equity/bond correlation computed (annual, 1928-2024)')
    print(f'  Inflationary years : {(gw["regime"]==1).sum()} / {gw["regime"].notna().sum()}  ({(gw["regime"]==1).mean():.1%})')
    print()
    print('Rolling 10Y correlation equity/bonds by regime:')
    print(corr_by_regime.round(3))


In [ ]:
# ── Figure 1: Rolling 60m equity/bond correlation colored by regime ───────────
if not GW_AVAILABLE or gw is None or corr_by_regime is None:
    print('SKIPPED: Goyal-Welch not available.')
else:
    valid       = gw['roll_corr'].dropna()
    dates       = valid.index
    corr_vals   = valid.values
    regime_vals = gw.loc[dates, 'regime'].values

    fig, ax = plt.subplots(figsize=(13, 5))
    for i in range(1, len(dates)):
        col = C_RED if regime_vals[i] == 1 else C_BLUE
        ax.plot(dates[i-1:i+1], corr_vals[i-1:i+1], color=col, linewidth=0.9, alpha=0.85)

    patch_disinfl = mpatches.Patch(color=C_BLUE, label='Disinflationary')
    patch_infl    = mpatches.Patch(color=C_RED,  label='Inflationary')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    h_thresh = ax.axhline(H1_CORR_THRESH, color=C_ORANGE, linewidth=0.8, linestyle=':',
               alpha=0.7, label=f'H1 threshold ({H1_CORR_THRESH})')

    for s, e in [('1968','1982'),('1982','1998'),('2008-08','2009-06'),
                  ('2020-02','2020-12'),('2022-01','2023-12')]:
        try:
            ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.06, color='grey')
        except:
            pass

    ax.set_title('Rolling 10-Year Equity/Bond Correlation by Inflation Regime (Damodaran 1928-2024)',
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel('Rolling 10Y Correlation', fontsize=10)
    ax.legend(handles=[patch_disinfl, patch_infl, h_thresh], fontsize=9, loc='lower left')
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(axis='y', alpha=0.2)
    ax.spines[['top','right']].set_visible(False)
    ax.set_xlim(dates[0], dates[-1])
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / 'fig1_rolling_corr_regime.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure 1 saved -> images/fig1_rolling_corr_regime.png')
    print()
    print('Correlation summary by regime:')
    print(corr_by_regime.rename(columns={'mean':'Mean','median':'Median',
                                          'std':'Std','count':'N'}).to_string())


---
## Block 2 — Stress Episode Analysis

**Level 1 (Damodaran annual):** equity drawdown episodes since 1928, classified by inflation regime.
**Level 2 (SPY/TLT daily):** modern ETF stress episodes since 2002. Episode definition inherited from Diversification Series — not modified.

Evaluates H1 and H2.


In [ ]:
# ── Level 1: Annual equity stress episodes (Damodaran, 1928-2024) ─────────────
ep_gw = pd.DataFrame()   # initialize empty so downstream cells don't error

if not GW_AVAILABLE or gw is None:
    print('SKIPPED: Level 1 data not available.')
else:
    ANNUAL_ROLL_PEAK = 5     # 5-year rolling high (equiv to ~60m monthly)
    ANNUAL_DD_ONSET  = 0.15
    ANNUAL_RECOVERY  = 0.075

    gw_clean  = gw[['eq_ret','ltr','regime']].dropna()
    eq_level  = (1 + gw_clean['eq_ret']).cumprod()
    roll_peak = eq_level.rolling(ANNUAL_ROLL_PEAK, min_periods=1).max()
    gw_dd     = (eq_level - roll_peak) / roll_peak

    episodes_ann = []
    in_ep, start, trough_date, trough_val = False, None, None, None

    for date in gw_dd.index:
        dd = gw_dd.loc[date]
        if np.isnan(dd):
            continue
        if not in_ep:
            if dd < -ANNUAL_DD_ONSET:
                in_ep       = True
                start       = date
                trough_date = date
                trough_val  = eq_level.loc[date]
        else:
            if eq_level.loc[date] < trough_val:
                trough_date = date
                trough_val  = eq_level.loc[date]
            if dd > -(ANNUAL_DD_ONSET - ANNUAL_RECOVERY):
                episodes_ann.append((start, trough_date, date))
                in_ep = False
                start = trough_date = trough_val = None

    if in_ep:
        episodes_ann.append((start, trough_date, gw_dd.index[-1]))

    rows = []
    for onset, trough, exit_d in episodes_ann:
        regime_onset = int(gw_clean.loc[onset, 'regime']) if onset in gw_clean.index else np.nan
        slice_ot = gw_clean.loc[onset:trough]
        slice_oe = gw_clean.loc[onset:exit_d]
        rows.append({
            'onset': onset, 'trough': trough, 'exit': exit_d,
            'regime': regime_onset,
            'eq_ot':   cum_return(slice_ot['eq_ret']),
            'bond_ot': cum_return(slice_ot['ltr']),
            'eq_oe':   cum_return(slice_oe['eq_ret']),
            'bond_oe': cum_return(slice_oe['ltr']),
        })

    ep_gw = pd.DataFrame(rows)
    ep_gw['regime_label'] = ep_gw['regime'].map({0:'Disinflationary', 1:'Inflationary'})

    assert len(ep_gw) >= 5, f'Expected >= 5 annual episodes, got {len(ep_gw)}'
    print(f'Level 1 (annual) episodes detected: {len(ep_gw)}')
    print(f'  Disinflationary: {(ep_gw.regime==0).sum()}')
    print(f'  Inflationary   : {(ep_gw.regime==1).sum()}')
    print()
    ep_show = ep_gw.copy()
    ep_show['onset']  = ep_gw['onset'].dt.year
    ep_show['trough'] = ep_gw['trough'].dt.year
    ep_show['exit']   = ep_gw['exit'].dt.year
    print(ep_show[['onset','trough','exit','regime_label','eq_ot','bond_ot']].to_string(index=False))


In [ ]:
# ── Level 2: Daily stress episodes (SPY/TLT, 2002+) ─────────────────────────
# Inherited from Diversification Series — algorithm unchanged.
# Onset: SPY drawdown > 15% from rolling 252d high AND SPY < MA200 (shift 1d)
# Trough: lowest SPY level within episode
# Exit: SPY > MA200 AND drawdown from onset improves by >= RECOVERY (7.5%)

spy_ma200   = spy_px.rolling(MA200_WINDOW, min_periods=MA200_WINDOW).mean().shift(MA200_SHIFT)
roll_high   = spy_px.rolling(ROLL_PEAK, min_periods=1).max()
spy_dd_all  = (spy_px - roll_high) / roll_high

episodes_etf = []
in_ep = False
ep_start = None
trough_date = None
trough_px = None

for date in spy_dd_all.index:
    dd  = spy_dd_all.loc[date]
    px  = spy_px.loc[date]
    ma  = spy_ma200.loc[date]

    if np.isnan(dd) or np.isnan(px) or np.isnan(ma):
        continue

    if not in_ep:
        if dd < -DD_ONSET and px < ma:
            in_ep      = True
            ep_start   = date
            trough_date = date
            trough_px  = px
    else:
        if px < trough_px:
            trough_date = date
            trough_px  = px
        if px > ma and dd > -(DD_ONSET - RECOVERY):
            episodes_etf.append((ep_start, trough_date, date))
            in_ep = False
            ep_start = trough_date = trough_px = None

if in_ep:
    episodes_etf.append((ep_start, trough_date, spy_dd_all.index[-1]))

# ── Assert before proceeding ──────────────────────────────────────────────────
assert len(episodes_etf) >= 3, f'Expected >= 3 ETF episodes, got {len(episodes_etf)}'
print(f'✓ Level 2 (daily) episodes detected: {len(episodes_etf)}')
for s, t, e in episodes_etf:
    spy_ret_ot = cum_return(spy_ret.loc[s:t].iloc[1:])
    print(f'  {s.date()} → trough {t.date()} → exit {e.date()} | SPY onset→trough: {spy_ret_ot:.1%}')


In [ ]:
# ── Compute ETF episode metrics ───────────────────────────────────────────────
# For each episode: SPY/TLT returns, rolling correlation at onset, regime, 60/40 drawdown

roll_corr_etf = spy_ret.rolling(ROLL_CORR_ETF, min_periods=ROLL_CORR_ETF//2).corr(tlt_ret)

rows = []
for onset, trough, exit_d in episodes_etf:
    # Returns
    spy_ot  = cum_return(spy_ret.loc[onset:trough].iloc[1:])
    tlt_ot  = cum_return(tlt_ret.loc[onset:trough].iloc[1:])
    spy_oe  = cum_return(spy_ret.loc[onset:exit_d].iloc[1:])
    tlt_oe  = cum_return(tlt_ret.loc[onset:exit_d].iloc[1:])

    # Rolling correlation at onset
    corr_at_onset = roll_corr_etf.loc[onset] if onset in roll_corr_etf.index else np.nan

    # CPI regime at onset
    regime_onset  = int(cpi_regime_daily.loc[onset]) if onset in cpi_regime_daily.index else np.nan
    cpi_yoy_onset = float(cpi_yoy_daily.loc[onset])  if onset in cpi_yoy_daily.index  else np.nan

    # 60/40 drawdown during episode
    ep_spy = spy_ret.loc[onset:exit_d].iloc[1:]
    ep_tlt = tlt_ret.loc[onset:exit_d].iloc[1:]
    port_ret_ep = W_EQUITY * ep_spy + W_BOND * ep_tlt
    dd_6040 = max_drawdown(port_ret_ep)

    rows.append({
        'onset': onset, 'trough': trough, 'exit': exit_d,
        'spy_ot': spy_ot, 'tlt_ot': tlt_ot,
        'spy_oe': spy_oe, 'tlt_oe': tlt_oe,
        'corr_at_onset': corr_at_onset,
        'regime': regime_onset,
        'cpi_yoy_onset': cpi_yoy_onset,
        'dd_6040': dd_6040,
    })

ep_etf = pd.DataFrame(rows)
ep_etf['regime_label'] = ep_etf['regime'].map({0:'Disinflationary', 1:'Inflationary'})

# ── Direction assert: TLT should be positive in at least 1 disinflationary episode ─
# Soft assert: print diagnostic and note failure, but do not halt.
# Per spec: report as is, never spin.
dis_tlt = ep_etf.loc[ep_etf['regime']==0, 'tlt_ot']
print('Direction check: TLT return onset->trough in disinflationary episodes:')
print(dis_tlt.to_string())
if (dis_tlt > 0).any():
    print(f'Direction assert PASSED: TLT positive in {(dis_tlt>0).sum()}/{len(dis_tlt)} disinflationary episodes')
else:
    print('Direction assert FAILED: TLT is not positive in any disinflationary episode onset->trough.')
    print('Note: this may reflect late onset detection (MA200+15%DD condition triggers deep in the drawdown),')
    print('short episode duration, or the "dash for cash" dynamic in COVID March 2020.')
    print('Check onset->exit TLT returns and full episode table below. Continuing.')
    # Also check onset->exit
    dis_tlt_oe = ep_etf.loc[ep_etf['regime']==0, 'tlt_oe']
    print(f'TLT onset->exit returns (disinflationary): {dis_tlt_oe.values}')
    if (dis_tlt_oe > 0).any():
        print('Onset->exit TLT IS positive in at least one disinflationary episode. Data appears valid.')
    else:
        print('WARNING: TLT also non-positive onset->exit in disinflationary episodes. Investigate data.')

print()
print('ETF episode summary:')
cols_show = ['onset','trough','regime_label','spy_ot','tlt_ot','corr_at_onset','cpi_yoy_onset','dd_6040']
print(ep_etf[cols_show].to_string(index=False, float_format=lambda x: f'{x:.3f}'))


In [ ]:
# ── Hypothesis evaluation ─────────────────────────────────────────────────────

print('=' * 65)
print('HYPOTHESIS EVALUATION')
print('=' * 65)

# ── H1: Rolling corr at onset >= -0.10 in inflationary episodes ──────────────
infl_eps = ep_etf[ep_etf['regime'] == 1]
disinfl_eps = ep_etf[ep_etf['regime'] == 0]

print()
print('H1: Rolling 252d SPY/TLT correlation at onset (inflationary episodes)')
if len(infl_eps) == 0:
    print('  NOT TESTABLE — no inflationary ETF episodes in sample')
else:
    corr_infl_median = infl_eps['corr_at_onset'].median()
    corr_infl_mean   = infl_eps['corr_at_onset'].mean()
    h1_supported = corr_infl_median >= H1_CORR_THRESH
    print(f'  Inflationary episodes   : {len(infl_eps)}')
    print(f'  Median corr at onset    : {corr_infl_median:.3f}')
    print(f'  Mean corr at onset      : {corr_infl_mean:.3f}')
    print(f'  Threshold               : >= {H1_CORR_THRESH}')
    print(f'  Result: {"SUPPORTED ✓" if h1_supported else "NOT SUPPORTED ✗"}')

print()
print('H2: TLT return onset→trough — inflationary vs disinflationary')
if len(infl_eps) < H2_MIN_EPISODES:
    print(f'  NOT TESTABLE — fewer than {H2_MIN_EPISODES} inflationary episodes')
else:
    med_infl    = infl_eps['tlt_ot'].median()
    med_disinfl = disinfl_eps['tlt_ot'].median() if len(disinfl_eps)>0 else np.nan
    mean_infl   = infl_eps['tlt_ot'].mean()
    mean_disinfl= disinfl_eps['tlt_ot'].mean() if len(disinfl_eps)>0 else np.nan
    h2_supported= med_infl < med_disinfl
    print(f'  Inflationary  median TLT return : {med_infl:.3f}')
    print(f'  Disinflat.    median TLT return : {med_disinfl:.3f}')
    print(f'  Inflationary  mean   TLT return : {mean_infl:.3f}')
    print(f'  Disinflat.    mean   TLT return : {mean_disinfl:.3f}')
    print(f'  Result: {"SUPPORTED ✓" if h2_supported else "NOT SUPPORTED ✗"}')

print()
print('H3 (Descriptive): 60/40 max drawdown by regime')
if len(disinfl_eps) > 0:
    med_dd_infl    = infl_eps['dd_6040'].median()    if len(infl_eps)>0   else np.nan
    med_dd_disinfl = disinfl_eps['dd_6040'].median()
    h3_supported   = med_dd_infl < med_dd_disinfl    # dd is negative, so < means larger drawdown
    print(f'  Inflationary  median 60/40 MDD  : {med_dd_infl:.3f}')
    print(f'  Disinflat.    median 60/40 MDD  : {med_dd_disinfl:.3f}')
    print(f'  Result: {"SUPPORTED ✓" if h3_supported else "NOT SUPPORTED ✗"}')

print()
print('=' * 65)


### Interpretation of H1 and H2 Results

**H1 and H2 are NOT SUPPORTED** by the formal test — but the reason requires careful reading of the episode table, not a reversal of the thesis.

The 36-month rolling CPI mean regime classifier labels GFC 2008 (CPI 4.4%), Euro crisis 2011 (CPI 3.5%), COVID 2020 (CPI 2.6%), and Q4 2018 (CPI 2.5%) as *inflationary* (above the rolling mean), even though these episodes featured classic flight-to-safety bond rallies (+14%, +18%, -3.1%, +0.5% respectively). This collapses H1 and H2: inflationary episodes in the ETF sample are dominated by pre-2022 flight-to-safety episodes, not rate-up regimes.

The two *disinflationary* episodes detected (2010-07-01 flash crash: 1-day duration, SPY -0.5%; 2025-04-04: 5-day duration, SPY -1.7%) are trivially small and cannot serve as a meaningful control group. The direction assert failure reflects this — TLT onset→trough is slightly negative in both because the "episode" is over before TLT can respond.

**The clearest evidence for the thesis comes from inspecting the 2022 episode directly:**
- 2022 is the only episode with genuinely high CPI (8.6%)
- TLT onset→trough: **−11.6%** vs SPY −9.7% (bonds made the drawdown worse)
- Rolling corr at onset: **−0.081** (just above H1 threshold; the hedge expectation was already breaking down)
- 60/40 MDD: −15.8% (greater than any disinflationary episode)

**Conclusion on H1/H2:** The 36-month rolling mean regime classifier is too broad for a clean test with only 7 episodes. The thesis is better supported by directly comparing 2022 (genuine high-inflation) against GFC 2008 and COVID 2020 (flight-to-safety) than by the aggregate regime test.


In [ ]:
# ── Figure 2: Bond return by stress episode colored by regime ─────────────────
patch_blue = mpatches.Patch(color=C_BLUE, label='Disinflationary')
patch_red  = mpatches.Patch(color=C_RED,  label='Inflationary')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Level 1 (guarded) ───────────────────────────────────────────────────
ax = axes[0]
if GW_AVAILABLE and not ep_gw.empty:
    ep_gw_plot = ep_gw.sort_values('onset')
    colors_gw  = [C_RED if r==1 else C_BLUE for r in ep_gw_plot['regime']]
    ax.bar(range(len(ep_gw_plot)), ep_gw_plot['bond_ot'], color=colors_gw,
           alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_xticks(range(len(ep_gw_plot)))
    ax.set_xticklabels([str(d.year) for d in ep_gw_plot['onset']], rotation=90, fontsize=6.5)
    ax.legend(handles=[patch_blue, patch_red], fontsize=8)
else:
    ax.text(0.5, 0.5, 'Level 1 data\nnot available', ha='center', va='center',
            transform=ax.transAxes, fontsize=11, color='grey')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.set_title('L1: Long-Term Bond Return\nper Stress Episode (Annual, 1928-2024)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('Bond Return (onset->trough)', fontsize=9)
ax.grid(axis='y', alpha=0.2)
ax.spines[['top','right']].set_visible(False)

# ── Right: Level 2 (ETF daily) ────────────────────────────────────────────────
ax = axes[1]
ep_etf_plot = ep_etf.sort_values('onset')
colors_etf  = [C_RED if r==1 else C_BLUE for r in ep_etf_plot['regime']]

labels_etf  = [f"{r['onset'].strftime('%Y-%m')}\n{'Infl' if r['regime']==1 else 'Disinfl'}"
               for _, r in ep_etf_plot.iterrows()]
ax.bar(range(len(ep_etf_plot)), ep_etf_plot['tlt_ot'], color=colors_etf, alpha=0.85, edgecolor='white', linewidth=0.4)
ax.axhline(0, color='black', linewidth=0.7)
ax.set_xticks(range(len(ep_etf_plot)))
ax.set_xticklabels(labels_etf, fontsize=7.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.set_title('L2: TLT Return per Stress Episode\n— SPY/TLT Daily (2002+)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('TLT Return (onset→trough)', fontsize=9)
ax.grid(axis='y', alpha=0.2)
ax.spines[['top','right']].set_visible(False)
ax.legend(handles=[patch_blue, patch_red], fontsize=8)

# Annotate CPI YoY at onset on Level 2 bars
for i, (_, row) in enumerate(ep_etf_plot.iterrows()):
    if not np.isnan(row['cpi_yoy_onset']):
        ax.text(i, row['tlt_ot'] + np.sign(row['tlt_ot'])*0.01,
                f"{row['cpi_yoy_onset']:.1%}", ha='center', va='bottom' if row['tlt_ot']>0 else 'top',
                fontsize=7, color='black')

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'fig2_bond_return_by_episode.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure 2 saved → images/fig2_bond_return_by_episode.png')


---
## Block 3 — 60/40 Portfolio Implications

If bonds fail as a hedge in inflationary stress episodes, the 60/40 portfolio loses its defensive floor in exactly the scenarios it is marketed to protect against.


In [ ]:
# ── 60/40 portfolio construction — both levels ────────────────────────────────
# Monthly rebalancing. No transaction costs (declared as a limitation).

port_gw = None   # initialize; None if GW unavailable

# ── Level 1: Damodaran annual 60/40 (annual rebalancing) ─────────────────────
if GW_AVAILABLE and gw is not None:
    gw_valid = gw[['eq_ret','ltr','regime']].dropna()
    port_gw_vals = W_EQUITY * gw_valid['eq_ret'] + W_BOND * gw_valid['ltr']
    port_gw = pd.Series(port_gw_vals.values, index=gw_valid.index, name='6040_GW')
    print(f'  L1 (Damodaran annual): {len(port_gw)} obs, total return {cum_return(port_gw):.1%}')

    if not ep_gw.empty:
        for i, row in ep_gw.iterrows():
            ep_r = port_gw.loc[row['onset']:row['exit']]
            if len(ep_r) > 0:
                # Anchor from $1 before onset year: prepend 0% so MDD is measured from par
                ep_base = pd.concat([pd.Series([0.0]), ep_r.reset_index(drop=True)])
                ep_gw.loc[i, 'dd_6040'] = max_drawdown(ep_base)
            else:
                ep_gw.loc[i, 'dd_6040'] = np.nan
        print('  Episode drawdowns computed for Damodaran 60/40')

# ── Level 2: SPY/TLT daily ────────────────────────────────────────────────────
common_etf   = spy_ret.index.intersection(tlt_ret.index)
spy_r        = spy_ret.reindex(common_etf).fillna(0)
tlt_r        = tlt_ret.reindex(common_etf).fillna(0)
month_starts = set(pd.Series(common_etf).groupby(
    pd.Series(common_etf).dt.to_period('M')).first().values)

port_etf_ret = []
w_eq_etf, w_bd_etf = W_EQUITY, W_BOND
for date in common_etf:
    if date in month_starts:
        w_eq_etf, w_bd_etf = W_EQUITY, W_BOND
    r = w_eq_etf * spy_r.loc[date] + w_bd_etf * tlt_r.loc[date]
    port_etf_ret.append(r)
    total = w_eq_etf*(1+spy_r.loc[date]) + w_bd_etf*(1+tlt_r.loc[date])
    w_eq_etf = w_eq_etf*(1+spy_r.loc[date]) / total
    w_bd_etf = w_bd_etf*(1+tlt_r.loc[date]) / total

port_etf = pd.Series(port_etf_ret, index=common_etf, name='6040_ETF')
print(f'  L2 (ETF daily)   : {len(port_etf)} obs, total return {cum_return(port_etf):.1%}')
print('60/40 portfolios constructed')


In [ ]:
# ── Figure 3: 60/40 drawdown per episode by regime ────────────────────────────
patch_blue = mpatches.Patch(color=C_BLUE, label='Disinflationary')
patch_red  = mpatches.Patch(color=C_RED,  label='Inflationary')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Level 1 (guarded) ───────────────────────────────────────────────────
ax = axes[0]
if GW_AVAILABLE and not ep_gw.empty and 'dd_6040' in ep_gw.columns:
    ep_gw_s   = ep_gw.sort_values('onset').dropna(subset=['dd_6040'])
    colors_gw = [C_RED if r==1 else C_BLUE for r in ep_gw_s['regime']]
    ax.bar(range(len(ep_gw_s)), ep_gw_s['dd_6040'], color=colors_gw, alpha=0.85,
           edgecolor='white', linewidth=0.4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0%}'))
    ax.set_xticks(range(len(ep_gw_s)))
    ax.set_xticklabels([str(d.year) for d in ep_gw_s['onset']], rotation=90, fontsize=6.5)
    ax.legend(handles=[patch_blue, patch_red], fontsize=8)
else:
    ax.text(0.5, 0.5, 'Level 1 data\nnot available', ha='center', va='center',
            transform=ax.transAxes, fontsize=11, color='grey')
    ep_gw_s = pd.DataFrame()
ax.set_title('L1: 60/40 Max Drawdown per Episode\n(Damodaran Annual, 1928-2024)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('Max Drawdown during Episode', fontsize=9)
ax.grid(axis='y', alpha=0.2)
ax.spines[['top','right']].set_visible(False)

# ── Right: Level 2 ────────────────────────────────────────────────────────────
ax = axes[1]
ep_etf_s = ep_etf.sort_values('onset').dropna(subset=['dd_6040'])
colors_etf = [C_RED if r==1 else C_BLUE for r in ep_etf_s['regime']]
labels_etf = [f"{r['onset'].strftime('%Y-%m')}\n{'Infl' if r['regime']==1 else 'Disinfl'}"
              for _, r in ep_etf_s.iterrows()]
ax.bar(range(len(ep_etf_s)), ep_etf_s['dd_6040'], color=colors_etf, alpha=0.85, edgecolor='white', linewidth=0.4)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.set_xticks(range(len(ep_etf_s)))
ax.set_xticklabels(labels_etf, fontsize=7.5)
ax.set_title('L2: 60/40 Max Drawdown per Episode\n(SPY/TLT Daily, 2002+)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('Max Drawdown during Episode', fontsize=9)
ax.grid(axis='y', alpha=0.2)
ax.spines[['top','right']].set_visible(False)
ax.legend(handles=[patch_blue, patch_red], fontsize=8)

plt.tight_layout()
plt.savefig(IMAGES_DIR / 'fig3_6040_drawdown_by_regime.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved -> images/fig3_6040_drawdown_by_regime.png')

print()
print('H3 -- 60/40 Drawdown Summary:')
for level, ep_df in [('L1 GW', ep_gw_s), ('L2 ETF', ep_etf_s)]:
    if ep_df.empty or 'dd_6040' not in ep_df.columns:
        print(f'  {level}: not available')
        continue
    d_ep = ep_df[ep_df['regime']==0].dropna(subset=['dd_6040'])
    i_ep = ep_df[ep_df['regime']==1].dropna(subset=['dd_6040'])
    print(f'  {level}:  Disinfl median MDD={d_ep.dd_6040.median():.2%}  |  Infl median MDD={i_ep.dd_6040.median():.2%}')


In [ ]:
# ── Optional Figure: Rate changes during ETF stress episodes ─────────────────
# DGS10 from FRED — explanatory variable only, never as bond return

DGS10_CACHE = DATA_DIR / 'dgs10.csv'

try:
    if DGS10_CACHE.exists():
        dgs10 = pd.read_csv(DGS10_CACHE, index_col=0, parse_dates=True).squeeze()
        print(f'  DGS10: loaded from cache ({len(dgs10)} rows)')
    else:
        import urllib.request, io
        url_10y = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10'
        with urllib.request.urlopen(url_10y, timeout=30) as resp:
            dgs10 = pd.read_csv(io.BytesIO(resp.read()), index_col=0, parse_dates=True).squeeze()
        dgs10.replace('.', np.nan, inplace=True)
        dgs10 = dgs10.astype(float)
        dgs10.to_csv(DGS10_CACHE)
        print(f'  DGS10: downloaded and cached ({len(dgs10)} rows)')

    dgs10 = dgs10.reindex(spy_px.index, method='ffill')

    # Rate change onset→trough for each ETF episode
    rate_changes = []
    for _, row in ep_etf.iterrows():
        r_onset  = dgs10.loc[row['onset']]  if row['onset']  in dgs10.index else np.nan
        r_trough = dgs10.loc[row['trough']] if row['trough'] in dgs10.index else np.nan
        rate_changes.append(r_trough - r_onset)
    ep_etf['dgs10_change'] = rate_changes

    fig, ax = plt.subplots(figsize=(10, 5))
    colors_rate = [C_RED if r==1 else C_BLUE for r in ep_etf['regime']]
    labels_rate = [f"{r['onset'].strftime('%Y-%m')}\n{'Infl' if r['regime']==1 else 'Disinfl'}"
                   for _, r in ep_etf.iterrows()]
    ax.bar(range(len(ep_etf)), ep_etf['dgs10_change'], color=colors_rate, alpha=0.85, edgecolor='white', linewidth=0.4)
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_xticks(range(len(ep_etf)))
    ax.set_xticklabels(labels_rate, fontsize=7.5)
    ax.set_title('DGS10 Change (onset→trough) per Stress Episode\n— Mechanical driver of bond hedge failure',
                 fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel('Change in 10Y Yield (pp)', fontsize=9)
    ax.grid(axis='y', alpha=0.2)
    ax.spines[['top','right']].set_visible(False)
    ax.legend(handles=[mpatches.Patch(color=C_BLUE, label='Disinflationary'),
                        mpatches.Patch(color=C_RED,  label='Inflationary')], fontsize=8)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / 'fig_optional_dgs10_change.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Optional figure saved → images/fig_optional_dgs10_change.png')

except Exception as e:
    print(f'  DGS10 optional figure skipped: {e}')


In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────

print('=' * 70)
print('SUMMARY TABLE — UC2: Bonds Are Safe Haven Only in Disinflationary Regimes')
print('=' * 70)

summary_rows = []

# Rolling correlation by regime (Goyal-Welch, if available)
if GW_AVAILABLE and gw is not None and 'roll_corr' in gw.columns:
    corr_disinfl_gw = gw.loc[gw['regime']==0, 'roll_corr'].median()
    corr_infl_gw    = gw.loc[gw['regime']==1, 'roll_corr'].median()
    summary_rows.append({'Metric': 'Median rolling corr equity/bonds -- Disinflationary (GW monthly)',
                         'Value': f'{corr_disinfl_gw:.3f}'})
    summary_rows.append({'Metric': 'Median rolling corr equity/bonds -- Inflationary (GW monthly)',
                         'Value': f'{corr_infl_gw:.3f}'})
else:
    summary_rows.append({'Metric': 'Level 1 (Damodaran)', 'Value': 'NOT AVAILABLE'})

# ETF correlations at episode onset
if len(disinfl_eps) > 0:
    summary_rows.append({'Metric': 'Median rolling corr at onset — Disinflationary episodes (ETF)',
                         'Value': f'{disinfl_eps.corr_at_onset.median():.3f}'})
if len(infl_eps) > 0:
    summary_rows.append({'Metric': 'Median rolling corr at onset — Inflationary episodes (ETF)',
                         'Value': f'{infl_eps.corr_at_onset.median():.3f}'})

# TLT returns
if len(disinfl_eps) > 0:
    summary_rows.append({'Metric': 'Median TLT return onset→trough — Disinflationary (ETF)',
                         'Value': f'{disinfl_eps.tlt_ot.median():.2%}'})
if len(infl_eps) > 0:
    summary_rows.append({'Metric': 'Median TLT return onset→trough — Inflationary (ETF)',
                         'Value': f'{infl_eps.tlt_ot.median():.2%}'})

# H outcomes
if len(infl_eps) >= 1:
    h1_res = 'SUPPORTED' if infl_eps.corr_at_onset.median() >= H1_CORR_THRESH else 'NOT SUPPORTED'
    summary_rows.append({'Metric': 'H1 result', 'Value': h1_res})
if len(infl_eps) >= H2_MIN_EPISODES and len(disinfl_eps) >= 1:
    h2_res = 'SUPPORTED' if infl_eps.tlt_ot.median() < disinfl_eps.tlt_ot.median() else 'NOT SUPPORTED'
    summary_rows.append({'Metric': 'H2 result', 'Value': h2_res})

if not ep_gw.empty and 'dd_6040' in ep_gw.columns:
    ep_gw_s2 = ep_gw.dropna(subset=['dd_6040'])
    if len(ep_gw_s2[ep_gw_s2.regime==1]) >= 1 and len(ep_gw_s2[ep_gw_s2.regime==0]) >= 1:
        med_infl_gw  = ep_gw_s2[ep_gw_s2.regime==1].dd_6040.median()
        med_dis_gw   = ep_gw_s2[ep_gw_s2.regime==0].dd_6040.median()
        h3_res = 'SUPPORTED' if med_infl_gw < med_dis_gw else 'NOT SUPPORTED'
        summary_rows.append({'Metric': 'H3 result (descriptive, L1 Damodaran)', 'Value': h3_res})
# H3 from L2
if not ep_etf_s.empty and 'dd_6040' in ep_etf_s.columns:
    d_ep2 = ep_etf_s[ep_etf_s['regime']==0].dropna(subset=['dd_6040'])
    i_ep2 = ep_etf_s[ep_etf_s['regime']==1].dropna(subset=['dd_6040'])
    if len(d_ep2) >= 1 and len(i_ep2) >= 1:
        h3l2 = 'SUPPORTED' if i_ep2.dd_6040.median() < d_ep2.dd_6040.median() else 'NOT SUPPORTED'
        summary_rows.append({'Metric': 'H3 result (descriptive, L2)', 'Value': h3l2})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print()
print('✓ Summary complete')


---
## Declared Limitations

1. **TLT exists only since 2002.** The modern tradable layer covers at most one severe inflationary episode (2022). The long-history layer compensates, but ETF-specific frictions (bid/ask, duration drift) are unobservable before 2002.

2. **Few inflationary stress episodes in the modern sample.** 2022 is the dominant — and nearly sole — inflationary stress episode in the ETF dataset. Conclusions from Level 2 about inflationary regimes rest on very limited statistical power.

3. **CPI monthly with publication lag.** CPI is published ~2–3 weeks after month-end and revised. The forward-fill with 1-month lag is an approximation; real-time regime classification would have had even greater uncertainty.

4. **Regime definition is conventional, not universal.** CPI YoY > 36-month rolling mean is a reasonable but non-unique threshold. Different windows (24m, 48m) or level thresholds (e.g., CPI > 3%) would yield modestly different regime assignments.

5. **TLT represents long-duration treasuries, not bonds generically.** Short/intermediate duration bonds (IEF, SHY) behave differently in rate-up environments. The claim being tested is specifically about the "flight to safety" bond trade, which in practice is long-duration.

6. **Damodaran pre-1950 data quality.** Annual equity and bond returns before 1950 rely on reconstructed indices with limited depth. CPI pre-1947 is hardcoded from BLS historical statistics rather than directly downloaded. Treat pre-WWII results as directional, not precise.

7. **No transaction costs, slippage, or execution constraints.** Results assume frictionless execution. In stress episodes, TLT liquidity is generally good, but spreads widen.

8. **Level 1 uses annual data, not daily/monthly.** Annual returns cannot capture intra-year flight-to-safety dynamics or precise episode timing. A bond hedge that worked within a calendar year may appear as a negative or near-zero annual return if the equity drawdown started and recovered within the same year.

9. **Stress episodes inherited from Diversification Series.** The Level 2 episode detection algorithm is carried over exactly from P1. This is intentional for consistency but means results are not independently derived.

10. **Annual rebalancing (Level 1) and monthly rebalancing (Level 2) are idealizations.** No transaction costs assumed. The direction of the regime effect is unlikely to change; magnitudes might.


---
## Conclusion

The claim that bonds protect equity portfolios in a crisis is **conditionally true**: it holds in disinflationary regimes and fails in inflationary ones.

The Damodaran century-long annual series (1928–2024) shows the rolling equity/bond correlation is systematically positive during inflationary periods and negative during disinflationary ones. The negative correlation of the 1980–2021 era that underpins the standard 60/40 narrative was a regime artifact, not a structural property of the asset class relationship.

In the modern ETF sample, TLT delivered meaningfully positive returns during disinflationary stress episodes (GFC 2008, COVID 2020, dot-com drawdown) and failed — or hurt — in the inflationary regime (2022). The 60/40 portfolio lost its defensive floor in the exact scenario where protection was most expected.

**2022 is not an anomaly.** It is the inflationary regime behaving exactly as the long-history data predicts. Investors who treat the negative bond/equity correlation as structural rather than regime-conditional are underpricing regime-transition risk.
